# Lab 06 - Integración de Datos y Streaming

**Objetivos:**
- Trabajar con diferentes formatos de datos
- Simular streaming con Auto Loader
- Integración de múltiples fuentes
- Best practices de almacenamiento

## Parte 1: Integración de Múltiples Formatos

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

print("🔧 Preparando datos de diferentes fuentes...")

In [ ]:
# =============================================================================
# FUENTE 1: CSV - CATÁLOGO DE PRODUCTOS
# =============================================================================
# CSV es común para: Datos de referencia, catálogos, exports de sistemas legacy
# Ventajas: Universal, legible por humanos, compatible con Excel
# Desventajas: Sin tipos de datos (todo es string), ineficiente para volúmenes grandes

# Generar 50 productos aleatorios
products_data = [
    {"product_id": f"P{i:03d}",  # ID con padding: P001, P002...
     "name": f"Product {i}",
     "category": random.choice(["Electronics", "Clothing", "Food"]),
     "price": round(random.uniform(10, 500), 2)}
    for i in range(1, 51)  # List comprehension: 50 productos
]

df_products = spark.createDataFrame(products_data)

# Tabla managed para productos
table_products = "lab06_source_products"

# Escribir como tabla Delta (mejor que CSV en DBFS)
df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_products)

print(f"✅ Productos guardados: {table_products}")

# 💡 OPCIONES COMUNES PARA CSV (si usas archivos externos):
# - header: true/false (incluir fila de encabezados)
# - delimiter: "," (default), "\t" (tab), "|" (pipe)
# - quote: '"' (carácter para valores con espacios/comas)
# - escape: '\\' (escapar caracteres especiales)
# - inferSchema: true (inferir tipos al leer, más lento pero conveniente)

In [ ]:
# =============================================================================
# FUENTE 2: JSON - TRANSACCIONES
# =============================================================================
# JSON es ideal para: APIs, datos semi-estructurados, eventos, logs
# Ventajas: Flexible (objetos anidados, arrays), mantiene tipos, legible
# Desventajas: Más pesado que Parquet, más lento de procesar

transactions_data = []
for i in range(200):
    transactions_data.append({
        "transaction_id": f"TXN{i:05d}",  # TXN00001, TXN00002...
        
        # Timestamp ISO 8601 (estándar universal)
        "timestamp": (datetime.now() - timedelta(hours=random.randint(0, 72))).isoformat(),
        
        "product_id": f"P{random.randint(1, 50):03d}",  # Referencias a productos
        "quantity": random.randint(1, 5),
        "customer_id": f"C{random.randint(1000, 9999)}"
    })

df_transactions = spark.createDataFrame(transactions_data)

# Tabla managed para transacciones
table_transactions = "lab06_source_transactions"

# Escribir como tabla Delta
df_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_transactions)

print(f"✅ Transacciones guardadas: {table_transactions}")

# 💡 FORMATOS JSON EN SPARK:
# - JSON Lines (default): Cada línea = 1 objeto JSON (paralelizable)
#   {"id":1,"name":"Alice"}
#   {"id":2,"name":"Bob"}
# - JSON único: 1 archivo con array [{},{}] (NO paralelizable, evitar)

# 📊 JSON vs CSV:
# JSON: Tipos preservados, objetos anidados, mejor para APIs
# CSV: Más simple, mejor para tablas planas, compatible con Excel

In [ ]:
# =============================================================================
# FUENTE 3: PARQUET - DATOS DE CLIENTES
# =============================================================================
# Parquet es ideal para: Big Data, data lakes, analytics
# Ventajas: Columnar (queries rápidas), compresión excelente, tipos preservados
# Desventajas: No legible por humanos, requiere herramientas especializadas

customers_data = [
    {"customer_id": f"C{i}", 
     "name": f"Customer {i}", 
     "email": f"customer{i}@example.com", 
     "country": random.choice(["US", "UK", "CA", "AU"])}
    for i in range(1000, 10000)
]

df_customers = spark.createDataFrame(customers_data)

# Tabla managed para clientes
table_customers = "lab06_source_customers"

# Escribir como tabla Delta (mejor que Parquet para data lakes modernos)
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_customers)

print(f"✅ Clientes guardados: {table_customers}")

## Parte 2: Leer Diferentes Formatos

In [ ]:
# Leer tabla de productos
df_products_read = spark.table("lab06_source_products")

print(f"📊 Productos: {df_products_read.count()} registros")
display(df_products_read.limit(5))

In [ ]:
# Leer tabla de transacciones
df_transactions_read = spark.table("lab06_source_transactions")

print(f"📊 Transacciones: {df_transactions_read.count()} registros")
display(df_transactions_read.limit(5))

In [ ]:
# Leer tabla de clientes
df_customers_read = spark.table("lab06_source_customers")

print(f"📊 Clientes: {df_customers_read.count()} registros")
display(df_customers_read.limit(5))

## Parte 3: Join y Enriquecimiento de Datos

In [ ]:
# =============================================================================
# JOIN: ENRIQUECIMIENTO DE DATOS
# =============================================================================
# Objetivo: Combinar transacciones con información de productos
# Patrón común: Datos transaccionales + datos de referencia

# 🔗 TIPOS DE JOIN EN SPARK:
# - "inner": Solo registros que hacen match en AMBOS lados (intersección)
# - "left" (left_outer): TODOS de la izquierda + matches de derecha
# - "right" (right_outer): TODOS de la derecha + matches de izquierda
# - "outer" (full_outer): TODOS de ambos lados (unión completa)
# - "left_semi": Solo izquierda que tiene match (como EXISTS en SQL)
# - "left_anti": Solo izquierda SIN match (como NOT EXISTS en SQL)
# - "cross": Producto cartesiano (cada fila izq × cada fila der, usar con cuidado!)

# ⚖️ ¿CUÁNDO USAR CADA TIPO?
# LEFT: Cuando quieres mantener todas las transacciones (más común)
# INNER: Solo transacciones con producto válido (más estricto)
# LEFT_ANTI: Encontrar transacciones sin producto (validación de integridad)

# Renombrar columnas de productos para evitar conflictos
df_products_renamed = df_products_read \
    .withColumnRenamed("name", "product_name") \
    .withColumnRenamed("category", "product_category")

# Join tipo LEFT (mantener TODAS las transacciones, incluso sin producto match)
df_enriched = df_transactions_read \
    .join(
        df_products_renamed,  # DataFrame de productos con nombres únicos
        "product_id",         # Columna de join (debe existir en ambos DataFrames)
        "left"                # Tipo de join
    ) \
    .withColumn(
        "total_amount",       # Nueva columna calculada
        col("quantity") * col("price")  # Cantidad × precio unitario
    )

print(f"✅ Datos enriquecidos: {df_enriched.count()} registros")
display(df_enriched.limit(10))

# 🚀 OPTIMIZACIÓN DE JOINS:
# - Broadcast Join: Si un DataFrame es pequeño (<10MB), usar broadcast(df_small)
#   Evita shuffle, mucho más rápido
# - Bucketing: Particionar ambos DataFrames por la misma columna de join
# - Salted Keys: Si hay skew (algunos valores muy frecuentes), agregar sal

In [ ]:
# Renombrar columnas de customers para evitar conflictos
df_customers_renamed = df_customers_read \
    .withColumnRenamed("name", "customer_name")

# Join con Customers
df_complete = df_enriched \
    .join(df_customers_renamed, "customer_id", "left") \
    .select(
        "transaction_id",
        "timestamp",
        "customer_id",
        "customer_name",
        "email",
        "country",
        "product_id",
        "product_name",
        "product_category",
        "quantity",
        "price",
        "total_amount"
    )

print(f"✅ Dataset completo: {df_complete.count()} registros")
display(df_complete.limit(10))

## Parte 4: Agregaciones y Análisis

In [ ]:
# Análisis por categoría
df_by_category = df_complete \
    .groupBy("product_category") \
    .agg(
        count("*").alias("num_transactions"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_transaction_amount"),
        sum("quantity").alias("total_units_sold"),
        countDistinct("customer_id").alias("unique_customers")
    ) \
    .orderBy(col("total_revenue").desc())

print("📊 Análisis por Categoría:")
display(df_by_category)

In [ ]:
# Análisis por país
df_by_country = df_complete \
    .groupBy("country") \
    .agg(
        count("*").alias("num_transactions"),
        sum("total_amount").alias("total_revenue"),
        countDistinct("customer_id").alias("unique_customers")
    ) \
    .orderBy(col("total_revenue").desc())

print("📊 Análisis por País:")
display(df_by_country)

In [ ]:
# Top 10 productos
df_top_products = df_complete \
    .groupBy("product_id", "product_name", "product_category") \
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .limit(10)

print("📊 Top 10 Productos:")
display(df_top_products)

## Parte 5: Guardar en Data Lakehouse

In [ ]:
# Guardar dataset completo particionado
table_complete = "lab06_transactions_complete"

df_complete \
    .withColumn("date", to_date("timestamp")) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("product_category", "country") \
    .saveAsTable(table_complete)

print(f"✅ Datos guardados: {table_complete}")

In [ ]:
# Guardar agregaciones (Serving Layer)

# Por categoría
df_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lab06_analytics_by_category")

# Por país
df_by_country.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lab06_analytics_by_country")

print("✅ Agregaciones guardadas en serving layer")

## Parte 6: Simular Streaming con Auto Loader

In [ ]:
# Preparar directorio para streaming
streaming_input = "/tmp/lab06/streaming/input"
streaming_checkpoint = "/tmp/lab06/streaming/checkpoint"
streaming_output = "/tmp/lab06/streaming/output"

# Limpiar directorios anteriores
dbutils.fs.rm(streaming_input, True)
dbutils.fs.rm(streaming_checkpoint, True)
dbutils.fs.rm(streaming_output, True)

# Crear archivo inicial
batch1 = spark.range(0, 100).withColumn("value", rand() * 1000)
batch1.write.format("json").mode("overwrite").save(f"{streaming_input}/batch1")

print(f"✅ Directorio streaming preparado")

In [ ]:
# Leer con Auto Loader (cloudFiles)
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", streaming_checkpoint) \
    .load(streaming_input)

# Transformación
df_stream_processed = df_stream \
    .withColumn("processed_at", current_timestamp()) \
    .withColumn("value_category",
                when(col("value") < 250, "Low")
                .when(col("value") < 750, "Medium")
                .otherwise("High"))

# Escribir stream
query = df_stream_processed.writeStream \
    .format("delta") \
    .option("checkpointLocation", streaming_checkpoint) \
    .outputMode("append") \
    .start(streaming_output)

print("🔄 Streaming iniciado...")
print(f"   Query ID: {query.id}")
print(f"   Status: {query.status}")

In [ ]:
# Agregar más archivos al stream (simular ingesta continua)
import time

for i in range(2, 5):
    batch = spark.range(0, 50).withColumn("value", rand() * 1000)
    batch.write.format("json").mode("overwrite").save(f"{streaming_input}/batch{i}")
    print(f"✅ Batch {i} agregado")
    time.sleep(2)

print("\n⏳ Esperando procesamiento...")
time.sleep(5)

In [ ]:
# Verificar resultados
df_stream_results = spark.read.format("delta").load(streaming_output)
print(f"📊 Registros procesados: {df_stream_results.count()}")

# Ver distribución por categoría
display(df_stream_results.groupBy("value_category").count().orderBy("value_category"))

In [ ]:
# Detener streaming
query.stop()
print("⏹️  Streaming detenido")

## Resumen del Laboratorio

### Competencias Adquiridas:

**Integración Multi-Formato:**
- Lectura y escritura de CSV, JSON y Parquet
- Configuración de opciones específicas por formato (header, delimiter, schema inference)
- Comparación de performance y casos de uso por formato

**Transformaciones de Datos:**
- JOINs complejos (left, inner) para enriquecimiento de datos
- Integración de datos transaccionales con dimensiones de referencia
- Cálculos derivados (total_amount = quantity × price)

**Optimización:**
- Particionamiento de datos por columnas clave (region, date)
- Estrategias de escritura eficiente para grandes volúmenes

**Streaming (Auto Loader):**
- Configuración de streaming ingestion con checkpoints
- Procesamiento incremental de datos
- Manejo de schema evolution

### Próximos Pasos:
- Implementar integración con Azure Storage Account (ADLS Gen2)
- Configurar Unity Catalog para governance de datos
- Explorar Apache Kafka para streaming en tiempo real
- Implementar pipelines de CI/CD con Databricks Repos

💡 **Próximos pasos**: Integrar con servicios Azure reales (Storage Account, Cosmos DB, Event Hubs) para producción.